# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring / Ranking** (implemented via a classification model)

My lane (Refresh / Content Opportunity Scoring) isn't really "does this page
belong to class A or B" in the way a typical classification problem is framed —
the actual deliverable is a **ranked queue**: an ordered list of pages, most
worth reviewing first. I get there by training a binary classifier
(page will decline vs. won't) and using its predicted *probability* as the
ranking score, rather than just its 0/1 label. This matches how the starter
pipeline works: `random_forest.predict_proba()` feeds the final ranked
`refresh_queue.csv`, not just a hard classification.

It's not clustering (I'm not grouping pages into unlabeled archetypes — that's
Lane 3), and it's not pure regression (I don't need an exact numeric value,
just a reliable ordering of "most likely to need review" to "least likely").

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy (what I've used so far):**
`is_declining_label = (trend_direction == "down")` — this is a **current-window
proxy**, not a future outcome. It's calculated from the same window I'd be
scoring, which the lane guide explicitly calls a "beginner proxy," not the
ideal capstone target.

**Stronger target I want to move toward:**
A genuine future-looking label built from the warehouse daily facts:
`features from prior 90 days -> decline over next 30 days`
i.e. did impressions/clicks drop by more than a threshold I define (e.g. >20%
month-over-month), measured strictly *after* the feature window ends — so the
feature window and the label window never overlap.

Where the label comes from: an **observed outcome** (real impressions/clicks
over time), not a rule FlyRank's product already computed — this keeps me from
just teaching a model to copy an existing decision flag.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@K** (specifically Precision@50)

Why this, not accuracy: a reviewer has limited capacity — say, 50 pages a week,
not 30,000. What matters isn't "is the model right about every page in the
dataset," it's "of the top 50 pages the model flags, how many are actually
worth reviewing?" That's exactly what Precision@K measures, and it's what the
starter results already reported: baseline 0.240 vs. random forest 0.740 at K=50.

**Secondary checks I'd also want:**
- Average precision, to judge the ranking quality beyond just the top 50
- A by-hand review of the actual top 20 pages, to sanity check the reason
  codes make sense to a human, not just the number
- Recall, if missing a real decline is more costly than reviewing a false
  positive — worth deciding explicitly, not assuming

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content item (content_id), not a client and not a single day.
# Show that clearly:
print(f"Total rows: {len(df):,}")
print(f"Unique content_id values: {df['content_id'].nunique():,}")
print(f"Unique client_id values: {df['client_id'].nunique():,}")
print("\nIf rows == unique content_id count, one row really is one content item:")
print(len(df) == df['content_id'].nunique())

# The columns that matter for my lane's target + a few key features
cols = ["content_id", "client_id", "trend_direction", "impressions_90d",
        "days_since_last_update", "avg_position", "ctr", "word_count"]
df[cols].head(10)

In [ ]:
# Sketch what the target column would look like (current proxy)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True).rename("share"))
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like `stale_visible_page`: days_since_last_update >= 180 AND
impressions_90d >= 500) can only combine a handful of thresholds a person
thought of in advance, with fixed, equal-feeling importance. Real pages don't
decline for one clean reason — staleness, position, CTR, word count, and query
concentration interact in ways that are hard to hand-write as clean if/else
logic. A page might be only mildly stale but also losing position AND getting
thin engagement — no single hard-coded rule captures that combination well,
but a model can learn how much each signal matters, together, from evidence.

This isn't a guess — it's already measured on this exact data: the starter's
hand-written rule scores 0.240 at Precision@50, while a random forest trained
on the same observable signals scores 0.740 — about 3x better at actually
finding pages worth reviewing first, under client-holdout validation (so it's
not just memorizing quirks of specific clients).

## Self-check

Before you submit, confirm each line honestly:

- [done] Every section above is filled — markdown thinking AND the code that backs it
- [done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done] No client names, URLs, or private queries anywhere
- [done] My claims use careful words: observed, measured, directional, decision-support
- [done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.